In [2]:
# Experiment 10: Encoder (BERT fill-mask) vs decoder (GPT-2 generation) vs encoder-decoder (Flan-T5) on the same tasks
import time, torch
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

bert = pipeline("fill-mask", model="bert-base-uncased")
gpt2 = pipeline("text-generation", model="gpt2")
t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").eval()

def run_bert(prompt):   # encoder: can only fill a [MASK] using left + right context
    return bert(prompt)[0]["token_str"]
def run_gpt2(prompt):   # decoder: continues text left-to-right
    out = gpt2(prompt, max_new_tokens=12, do_sample=False, pad_token_id=50256)[0]["generated_text"]
    return out[len(prompt):].strip().split("\n")[0]
def run_t5(prompt):     # encoder-decoder: reads the whole input, writes a new output (instruction following)
    ids = t5_tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        out = t5.generate(ids, max_new_tokens=20)
    return t5_tok.decode(out[0], skip_special_tokens=True)

# Each task phrased the natural way for each family
tasks = [
 ("Fill the blank",  "The capital of France is [MASK].", "The capital of France is", "Fill in the blank: The capital of France is ___."),
 ("Factual QA",      "Question: what is the largest planet? Answer: [MASK].", "Q: What is the largest planet?\nA:", "What is the largest planet?"),
 ("Sentiment",       "The movie was wonderful. Overall it was [MASK].", "Review: The movie was wonderful. Sentiment:", "Is this review positive or negative? The movie was wonderful."),
 ("Translation",     "In German, hello is [MASK].", "English: Good morning\nGerman:", "Translate English to German: Good morning"),
 ("Summarise",       "Transformers use attention. In short, they rely on [MASK].", "Transformers use self-attention to relate every token to every other token, which made RNNs obsolete. TL;DR:", "Summarize: Transformers use self-attention to relate every token to every other token, which made RNNs obsolete."),
]
rows = []
for name, p_bert, p_gpt, p_t5 in tasks:
    row = {"task": name}
    for fam, fn, p in [("BERT (encoder)", run_bert, p_bert), ("GPT-2 (decoder)", run_gpt2, p_gpt), ("Flan-T5 (enc-dec)", run_t5, p_t5)]:
        t = time.time(); row[fam] = fn(p); row[fam + " s"] = round(time.time() - t, 2)
    rows.append(row)

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 45)
print("Comparison table:")
print(df[["task", "BERT (encoder)", "GPT-2 (decoder)", "Flan-T5 (enc-dec)"]].to_string(index=False))
print("\nLatency (s):")
print(df[["task", "BERT (encoder) s", "GPT-2 (decoder) s", "Flan-T5 (enc-dec) s"]].to_string(index=False))
print("\nTakeaway: BERT = understanding / single-slot filling; GPT-2 = free continuation (no instruction tuning);")
print("Flan-T5 = instruction-tuned seq2seq, best for task-style inputs like translate / summarise / classify.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

... (19 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Comparison table:
          task BERT (encoder)                                            GPT-2 (decoder)                             Flan-T5 (enc-dec)
Fill the blank          paris the capital of the French Republic, and the capital of the                                        london
    Factual QA           mars    The largest planet is the Sun. It is the largest planet                                        uranus
     Sentiment           good                                                       5/10                                      positive
   Translation          hello                                                      Hello                                 Good morning!
     Summarise      attention              RNNs are a great way to learn about the world Transformers are a relic of the original RNN.

Latency (s):
          task  BERT (encoder) s  GPT-2 (decoder) s  Flan-T5 (enc-dec) s
Fill the blank              0.30               0.97                 0.57
    Factual 

In [ ]:
# Results (printed separately so download logs do not hide them)
print(df[["task", "BERT (encoder)", "GPT-2 (decoder)", "Flan-T5 (enc-dec)"]].to_string(index=False))
print()
print(df[["task", "BERT (encoder) s", "GPT-2 (decoder) s", "Flan-T5 (enc-dec) s"]].to_string(index=False))

          task BERT (encoder)                                            GPT-2 (decoder)                             Flan-T5 (enc-dec)
Fill the blank          paris the capital of the French Republic, and the capital of the                                        london
    Factual QA           mars    The largest planet is the Sun. It is the largest planet                                        uranus
     Sentiment           good                                                       5/10                                      positive
   Translation          hello                                                      Hello                                 Good morning!
     Summarise      attention              RNNs are a great way to learn about the world Transformers are a relic of the original RNN.

          task  BERT (encoder) s  GPT-2 (decoder) s  Flan-T5 (enc-dec) s
Fill the blank              3.28               1.98                 0.55
    Factual QA              0.14           